In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# Making dataframes for all the cleaned csv files

ratings_df = pd.read_csv('ratings.csv')
reviews_df = pd.read_csv('reviews.csv')
subscribers_df = pd.read_csv('subscribers.csv')
titles_df = pd.read_csv('titles.csv')
watch_history_df = pd.read_csv('watch_history.csv')
watchlist_df = pd.read_csv('watchlist.csv')

In [0]:
ratings_df.info()

In [0]:
# From ratings.csv convert the rating_date type from object to datetime
ratings_df['rating_date'] = pd.to_datetime(ratings_df['rating_date'])

In [0]:
ratings_df.info()

In [0]:
reviews_df.info()

In [0]:
# from reviews_df convert the review_date from object to datetime
reviews_df['review_date'] = pd.to_datetime(reviews_df['review_date'])
reviews_df.info()

In [0]:
subscribers_df.info()

In [0]:
# from subscribers.csv convert the signup_date and churn_date from object to datetime
subscribers_df['signup_date'] = pd.to_datetime(subscribers_df['signup_date'])
subscribers_df['churn_date'] = pd.to_datetime(subscribers_df['churn_date'])

subscribers_df.info()

In [0]:
titles_df.info()

In [0]:
# in titles_df the date_added and license_expiry field should be of datetime type
titles_df['date_added'] = pd.to_datetime(titles_df['date_added'])
titles_df['license_expiry'] = pd.to_datetime(titles_df['license_expiry'])

titles_df.info()

In [0]:
watch_history_df.info()

In [0]:
# in watch_history_df, watch_date should of datetime type
watch_history_df['watch_date'] = pd.to_datetime(watch_history_df['watch_date'])

watch_history_df.info()

In [0]:
watchlist_df.info()

In [0]:
# in watchlist_df, added_date should be of datetime type
watchlist_df['added_date'] = pd.to_datetime(watchlist_df['added_date'])

watchlist_df.info()

# Monthly viewing volumne - a bar or a line chart showing number of sessions per month across the timeline. Highlight the busiest months.

In [0]:
# Using the watch_history_df we will create a chart against watch_date and take out the month part from that and then count the number of watch_id for that month and create a bar chart using matplotlib

import matplotlib.dates as mdates
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import pandas as pd

# 1. Extract the monthly period directly from the datetime watch_date column
watch_history_df['month'] = watch_history_df['watch_date'].dt.to_period('M')

# 2. Count watch_id occurrences per month across the timeline
monthly_volume = (
    watch_history_df.groupby('month')['watch_id']
    .count()
    .reset_index(name='session_count')
)

# Convert Period to Timestamp for matplotlib date formatting on the x-axis
monthly_volume['month_date'] = monthly_volume['month'].dt.to_timestamp()

# 3. Identify the busiest months (e.g., top 15% volume threshold)
threshold = monthly_volume['session_count'].quantile(0.85)
colors = [
    '#e63946' if count >= threshold else '#457b9d'
    for count in monthly_volume['session_count']
]

# 4. Create the bar chart
fig, ax = plt.subplots(figsize=(15, 6))

ax.bar(
    monthly_volume['month_date'],
    monthly_volume['session_count'],
    width=22,
    color=colors,
    edgecolor='none',
)

# 5. Format axes and timeline intervals (ticks every 6 months)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45, ha='right')

# Labels and styling
ax.set_title(
    'Monthly Viewing Volume Across Timeline',
    fontsize=14,
    fontweight='bold',
    pad=15,
)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Number of Sessions (watch_id count)', fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Custom legend to highlight busiest months
legend_elements = [
    Patch(
        facecolor='#e63946',
        label=f'Busiest Months (Top 15%, >{int(threshold):,} sessions)',
    ),
    Patch(facecolor='#457b9d', label='Regular Months'),
]
ax.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.show()

## Insights for "Monthly Viewing Volume Across Timeline"

_The Monthly Viewing Volume chart reveals a dramatic exponential growth in user engagement, with viewing sessions remaining minimal (near zero) from 2017 through mid-2024, then surging rapidly in late 2024 through 2026. The busiest months (top 15%, highlighted in red) are all concentrated in the final year of the timeline, with the peak occurring in late 2026 at over 200,000 sessions—more than 20 times the volume of just a year prior. This pattern suggests the platform experienced explosive subscriber growth and adoption during 2025-2026, possibly driven by content expansion, marketing campaigns, or market penetration strategies._

# Monthly watch hours trend - same as above but using watch_duration_min (converted to hours). Compare year-over-year.

In [0]:

import calendar
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd

# 1. Convert watch duration to hours and extract year/month components
watch_history_df['watch_hours'] = watch_history_df['watch_duration_min'] / 60.0
watch_history_df['year'] = watch_history_df['watch_date'].dt.year
watch_history_df['month'] = watch_history_df['watch_date'].dt.month

# 2. Aggregate monthly watch hours per year (Jan-Dec comparison)
yoy_df = (
    watch_history_df.groupby(['year', 'month'])['watch_hours']
    .sum()
    .unstack(level=0)  # Rows: months (1-12), Columns: years
)

# 3. Create the YoY comparison line chart
fig, ax = plt.subplots(figsize=(13, 6.5))

# Plot prior baseline years
for yr in yoy_df.columns:
    if yr < 2024:
        ax.plot(
            yoy_df.index,
            yoy_df[yr],
            linestyle='--',
            alpha=0.35,
            color='#6c757d',
            label=f'{yr}' if yr in [2021, 2022, 2023] else None,
        )

# Highlight high-volume / business growth years
ax.plot(
    yoy_df.index,
    yoy_df[2024],
    marker='^',
    linewidth=1.8,
    color='#2a9d8f',
    label='2024',
)
ax.plot(
    yoy_df.index,
    yoy_df[2025],
    marker='s',
    linewidth=2.2,
    color='#f4a261',
    label='2025',
)
ax.plot(
    yoy_df.index,
    yoy_df[2026],
    marker='o',
    linewidth=3.0,
    color='#e63946',
    label='2026 (Peak Business Year)',
)

# 4. Format X and Y axes
ax.set_xticks(range(1, 13))
ax.set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)], fontsize=11)


def hours_formatter(x, pos):
    if x >= 1e6:
        return f'{x*1e-6:.1f}M hrs'
    elif x >= 1e3:
        return f'{x*1e-3:.0f}K hrs'
    return f'{int(x)} hrs'


ax.yaxis.set_major_formatter(ticker.FuncFormatter(hours_formatter))

# Styling
ax.set_title(
    'Monthly Watch Hours Trend — Year-over-Year (YoY) Comparison',
    fontsize=14,
    fontweight='bold',
    pad=15,
)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Total Watch Hours', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(title='Timeline Year', frameon=True, loc='upper left')

plt.tight_layout()
plt.show()

## Insights for "Monthly Watch Hours Trend - Year-Over-Year (YoY) Comparison" 

_The year-over-year comparison reveals that watch hours remained negligible from 2017-2023 (baseline years shown as flat dashed lines near zero), with meaningful growth only beginning in 2024 and accelerating dramatically through 2025 and 2026. The 2026 peak business year exhibits a steep climb to over 1.0 million watch hours in May, representing explosive growth that far outpaces all prior years combined. The 2026 trend line also shows distinct seasonality, with the highest engagement concentrated in the first half of the year (January-May) before moderating—though still maintaining substantially higher levels than 2025—through the second half._

# Genre - wise watch hour share - a pie or a horizontal bar chart. Which genre earns the most watch time

In [0]:
# 1. Map genre from titles_df and calculate watch hours
genre_hours = (
    watch_history_df.merge(
        titles_df[['title_id', 'primary_genre']], on='title_id', how='left'
    )
    .assign(watch_hours=lambda df: df['watch_duration_min'] / 60.0)
    .groupby('primary_genre')['watch_hours']
    .sum()
    .sort_values(ascending=False)
)

# 2. Group top genres and combine the rest into "Other Genres"
top_n = 6
top_genres = genre_hours.head(top_n)
others = pd.Series({'Other Genres': genre_hours.iloc[top_n:].sum()})
pie_data = pd.concat([top_genres, others])

# 3. Highlight the top-performing genre
explode = [0.08 if genre == pie_data.index[0] else 0 for genre in pie_data.index]
colors = [
    '#e63946',
    '#457b9d',
    '#2a9d8f',
    '#e76f51',
    '#f4a261',
    '#a8dadc',
    '#d3d3d3',
]

# 4. Plot pie chart
plt.figure(figsize=(8, 8))
plt.pie(
    pie_data,
    labels=pie_data.index,
    autopct='%1.1f%%',
    startangle=140,
    explode=explode,
    colors=colors,
    pctdistance=0.8,
    textprops={'fontsize': 10},
)

plt.title('Genre-wise Watch Hour Share', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

## Insights for "Genre - Wise watch hour share"

_Drama leads as the most-watched genre with 15.0% of total watch hours, followed closely by Comedy (12.8%) and Action (11.8%), indicating a preference for emotionally engaging and high-energy content among subscribers. The top six individual genres collectively account for 64% of viewing time, while "Other Genres" represents a substantial 36%, suggesting the platform maintains a diverse content catalog that caters to varied audience interests beyond the mainstream categories. The relatively balanced distribution across multiple genres indicates a healthy content mix rather than over-reliance on any single genre to drive engagement._

# Content type split - a donut chart showing the share of hours for Movies vs TV Shows.

In [0]:
# 1. Map content 'type' from titles_df and convert watch duration to hours
merged_df = watch_history_df.merge(
    titles_df[['title_id', 'type']], on='title_id', how='left'
)
merged_df['watch_hours'] = merged_df['watch_duration_min'] / 60.0

# 2. Aggregate total watch hours by content type
type_hours = (
    merged_df.groupby('type')['watch_hours']
    .sum()
    .reset_index(name='total_watch_hours')
)

# 3. Create the Donut Chart
fig, ax = plt.subplots(figsize=(7, 7))

colors = ['#2a9d8f', '#e76f51']  # Movie: Teal, TV Show: Coral

wedges, texts, autotexts = ax.pie(
    type_hours['total_watch_hours'],
    labels=type_hours['type'],
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    pctdistance=0.75,
    textprops={'fontsize': 12, 'fontweight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)

# 4. Cut out the center circle to form the donut
centre_circle = plt.Circle((0, 0), 0.55, fc='white')
ax.add_artist(centre_circle)

# Display total hours in the center of the donut
total_hrs = type_hours['total_watch_hours'].sum()
ax.text(
    0,
    0,
    f'Total Hours\n{total_hrs/1e6:.2f}M',
    ha='center',
    va='center',
    fontsize=12,
    fontweight='bold',
    color='#333333',
)

ax.set_title(
    'Content Type Split: Movies vs TV Shows\n(Share of Watch Hours)',
    fontsize=14,
    fontweight='bold',
    pad=15,
)

plt.tight_layout()
plt.show()

## Insights for "Content Type Split: Movies vs TV Shows (Share of Watch Hours)"

_TV Shows overwhelmingly dominate the platform's viewing landscape, capturing 85.4% of total watch hours compared to just 14.6% for Movies—a nearly 6:1 ratio that reveals a strong subscriber preference for episodic and serialized content. This dramatic skew toward TV Shows suggests that the platform's engagement strategy is heavily driven by long-form series that enable binge-watching behavior and sustained viewer retention across multiple episodes. The 3.01 million total watch hours concentrate primarily in TV content, indicating that content acquisition and recommendation strategies should continue prioritizing serialized storytelling to align with demonstrated user preferences._

# Top 10 countries by watch hours - a horizontal bar chart

In [0]:
# 1. Map country from titles_df and convert watch duration to hours
country_hours = (
    watch_history_df.merge(
        titles_df[['title_id', 'country']], on='title_id', how='left'
    )
    .assign(watch_hours=lambda df: df['watch_duration_min'] / 60.0)
    .groupby('country')['watch_hours']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

# 2. Sort ascending for a top-down horizontal bar layout
plot_data = country_hours.sort_values(by='watch_hours', ascending=True)

# 3. Create Horizontal Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    plot_data['country'],
    plot_data['watch_hours'] / 1e3,  # in thousands (K)
    color='#1d3557',
    edgecolor='none',
    height=0.65,
)

# Highlight the #1 country with an accent color
bars[-1].set_color('#e63946')

# 4. Add data labels to each bar
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + 5,
        bar.get_y() + bar.get_height() / 2,
        f'{width:.1f}K',
        va='center',
        ha='left',
        fontsize=10,
        color='#222222',
    )

# 5. Styling and labels
ax.set_title(
    'Top 10 Countries by Watch Hours', fontsize=14, fontweight='bold', pad=15
)
ax.set_xlabel('Watch Hours (Thousands)', fontsize=11)
ax.set_ylabel('Country', fontsize=11)
ax.grid(axis='x', linestyle='--', alpha=0.5)

# Add right padding so labels fit cleanly
ax.set_xlim(0, (plot_data['watch_hours'].max() / 1e3) * 1.08)

plt.tight_layout()
plt.show()

## Insights for "Top 10 countries by watch hours" 

_The United States dominates global viewership with 741.6K watch hours—nearly 80% more than second-place India (412.1K) and representing almost as much watch time as the next three countries combined (UK, South Korea, and Japan). A clear geographic concentration emerges, with the top two English-speaking and Asian markets (US and India) driving the majority of engagement, while the remaining eight countries show a more gradual decline from 253K to 99.9K hours. This distribution suggests that while the platform has achieved meaningful penetration across diverse international markets spanning North America, Europe, Asia, and Latin America, the US market remains the critical anchor for overall platform engagement and likely drives content strategy and monetization priorities._

# Subscriber plan distribution - a pie chart. How many Basic vs Standard vs Premium subscribers?

In [0]:
# 1. Count subscribers per plan type (column name is 'plan_type')
plan_counts = subscribers_df['plan_type'].value_counts()

# 2. Format labels to display both percentage and exact counts
def make_autopct(values):
  def my_autopct(pct):
    total = sum(values)
    val = int(round(pct * total / 100.0))
    return f'{pct:.1f}%\n({val:,})'

  return my_autopct


# 3. Create the Pie Chart
fig, ax = plt.subplots(figsize=(8, 8))

colors = ['#457b9d', '#2a9d8f', '#e63946']
explode = (0.02, 0.02, 0.04)

wedges, texts, autotexts = ax.pie(
    plan_counts,
    labels=plan_counts.index,
    autopct=make_autopct(plan_counts),
    startangle=140,
    colors=colors,
    explode=explode,
    pctdistance=0.7,
    textprops={'fontsize': 11},
)

# Style percentage/count text inside slices
for autotext in autotexts:
  autotext.set_color('white')
  autotext.set_fontweight('bold')

ax.set_title(
    'Subscriber Plan Distribution', fontsize=14, fontweight='bold', pad=15
)
plt.tight_layout()
plt.show()

## Insights for "Subscriber Plan Distribution" 

_The Standard plan dominates the subscriber base with 40.8% (5,915 subscribers), making it the clear preference among users seeking a balance between features and cost. The Basic with Ads (29.8%, 4,324 subscribers) and Premium (29.5%, 4,275 subscribers) plans are virtually tied, each capturing approximately 30% of the market and revealing a nearly even split between price-conscious subscribers willing to tolerate ads and premium users seeking the full ad-free experience. This relatively balanced three-way distribution suggests the platform has successfully tiered its offerings to capture diverse subscriber segments across different willingness-to-pay levels, rather than over-relying on any single plan type for revenue._

# Device usage - a bar chart showing which devices people watch on the most.

In [0]:

import matplotlib.pyplot as plt
import pandas as pd

# 1. Count viewing sessions per device from watch_history_df
device_counts = watch_history_df['device'].value_counts()

# 2. Create the Bar Chart
fig, ax = plt.subplots(figsize=(9, 5.5))

# Highlight the most-watched device in red
colors = [
    '#e63946' if d == device_counts.index[0] else '#1d3557'
    for d in device_counts.index
]
bars = ax.bar(
    device_counts.index,
    device_counts.values,
    color=colors,
    width=0.6,
    edgecolor='none',
)

# 3. Add data labels above each bar
for bar in bars:
  yval = bar.get_height()
  ax.text(
      bar.get_x() + bar.get_width() / 2,
      yval + 3500,
      f'{yval:,}',
      ha='center',
      va='bottom',
      fontsize=10,
      fontweight='bold',
      color='#333333',
  )

# 4. Styling and labels
ax.set_title(
    'Device Usage: Number of Viewing Sessions by Device',
    fontsize=14,
    fontweight='bold',
    pad=15,
)
ax.set_xlabel('Device', fontsize=11)
ax.set_ylabel('Number of Sessions (watch_id count)', fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.set_ylim(0, max(device_counts.values) * 1.12)
plt.xticks(fontsize=10)

plt.tight_layout()
plt.show()

## Insights for "Device Usage: Number of Viewing Sessions by Device" 

_Smart TV dominates as the preferred viewing platform with 213,300 sessions—26% more than second-place Mobile (169,285) and representing over one-third of all viewing activity across devices. A clear device hierarchy emerges, with traditional large-screen options (Smart TV, Laptop) commanding 306,168 combined sessions, while mobile and portable devices (Mobile, Tablet) capture 232,184 sessions, indicating that viewers favor the home theater experience but maintain substantial on-the-go engagement. The relatively modest usage of Game Consoles (50,427) and Streaming Sticks (37,551) suggests these secondary devices serve niche use cases rather than primary viewing methods, reinforcing that Smart TV and Mobile are the critical platforms for content delivery optimization and user experience investment._

# Age distribution of subscribers - a histogram. Which age group watches the most?

Since it is not mentioned which age groups do I have to take, and the age is between 13-85, I am grouping the ages between "13-25", "25-40", "40-60" and "60-85".

In [0]:
# 1. Define custom age bins and labels
bins = [13, 25, 40, 60, 85]
labels = ['13-25', '25-40', '40-60', '60-85']

# 2. Assign age groups in subscribers_df
subscribers_df['age_group'] = pd.cut(
    subscribers_df['age'],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True,
)

# 3. Merge with watch_history_df to aggregate watch duration
merged_df = watch_history_df.merge(
    subscribers_df[['subscriber_id', 'age', 'age_group']],
    on='subscriber_id',
    how='left',
)
merged_df['watch_hours'] = merged_df['watch_duration_min'] / 60.0

# Watch time aggregation per group
age_watch_summary = (
    merged_df.groupby('age_group', observed=False)['watch_hours']
    .sum()
    .reindex(labels)
)

# 4. Plot Histogram of Subscriber Ages & Watch Hours Breakdown
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# --- Plot 1: Histogram of Subscriber Age Distribution ---
n, edges, patches = ax1.hist(
    subscribers_df['age'],
    bins=bins,
    color='#457b9d',
    edgecolor='white',
    linewidth=1.5,
    rwidth=0.95,
)

# Highlight the largest cohort
max_idx = np.argmax(n)
patches[max_idx].set_facecolor('#e63946')

ax1.set_title('Subscriber Age Distribution', fontsize=13, fontweight='bold', pad=12)
ax1.set_xlabel('Age', fontsize=11)
ax1.set_ylabel('Number of Subscribers', fontsize=11)
ax1.set_xticks(bins)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# Add counts on top of histogram bars
for count, edge_l, edge_r in zip(n, bins[:-1], bins[1:]):
  mid = (edge_l + edge_r) / 2
  ax1.text(
      mid,
      count + 120,
      f'{int(count):,}',
      ha='center',
      va='bottom',
      fontsize=10,
      fontweight='bold',
  )

# --- Plot 2: Total Watch Hours by Age Group ---
bars = ax2.bar(
    age_watch_summary.index,
    age_watch_summary.values / 1e3,  # in thousands (K)
    color=[
        '#e63946' if group == '25-40' else '#1d3557'
        for group in age_watch_summary.index
    ],
    width=0.55,
)

for bar in bars:
  h = bar.get_height()
  ax2.text(
      bar.get_x() + bar.get_width() / 2,
      h + 25,
      f'{h:.1f}K hrs',
      ha='center',
      va='bottom',
      fontsize=10,
      fontweight='bold',
  )

ax2.set_title(
    'Total Watch Hours by Age Group', fontsize=13, fontweight='bold', pad=12
)
ax2.set_xlabel('Age Group', fontsize=11)
ax2.set_ylabel('Watch Hours (Thousands)', fontsize=11)
ax2.grid(axis='y', linestyle='--', alpha=0.5)
ax2.set_ylim(0, (age_watch_summary.max() / 1e3) * 1.15)

plt.suptitle(
    'Subscriber Age Demographics & Viewing Time Analysis',
    fontsize=15,
    fontweight='bold',
    y=0.98,
)
plt.tight_layout()
plt.show()

## Insights for "Subscriber Age Demographics & Viweing Time Anlaysis"

_The 25-40 age group dominates both subscriber count (6,169, or 42.6% of the base) and total viewing engagement with 1,257.4K watch hours—approximately double the watch time of the youngest cohort (13-25) despite having only 2.1× the subscriber count, indicating higher per-capita engagement. The platform's core audience spans adults aged 25-60, collectively representing 78.9% of subscribers and 76.4% of total watch hours, while the 60-85 demographic remains significantly underrepresented with just 486 subscribers and 75.8K hours. Notably, the strong correlation between subscriber count and watch hours across age groups suggests that acquisition and retention challenges are consistent across demographics, with engagement levels generally proportional to cohort size rather than revealing age-specific retention issues._

# Completion rate by genre - bar chart. Which genre gets finished first?

In [0]:
# 1. Merge watch_history_df with titles_df to get primary_genre
merged_df = watch_history_df.merge(
    titles_df[['title_id', 'primary_genre']], on='title_id', how='left'
)

# 2. Calculate mean completion_pct for each genre
genre_completion = (
    merged_df.groupby('primary_genre')['completion_pct']
    .mean()
    .sort_values(ascending=False)
    .reset_index(name='avg_completion_pct')
)

# 3. Create the Bar Chart
fig, ax = plt.subplots(figsize=(12, 6))

top_genre = genre_completion.iloc[0]['primary_genre']
colors = [
    '#e63946' if g == top_genre else '#1d3557'
    for g in genre_completion['primary_genre']
]

bars = ax.bar(
    genre_completion['primary_genre'],
    genre_completion['avg_completion_pct'],
    color=colors,
    width=0.6,
    edgecolor='none',
)

# Add percentage values above each bar
for bar in bars:
  h = bar.get_height()
  ax.text(
      bar.get_x() + bar.get_width() / 2,
      h + 0.8,
      f'{h:.1f}%',
      ha='center',
      va='bottom',
      fontsize=9,
      fontweight='bold',
      color='#333333',
  )

# Styling and labels
ax.set_title(
    'Average Completion Rate by Genre (%)',
    fontsize=14,
    fontweight='bold',
    pad=15,
)
ax.set_xlabel('Genre', fontsize=11)
ax.set_ylabel('Average Completion (%)', fontsize=11)
ax.set_ylim(0, 80)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right', fontsize=10)

plt.tight_layout()
plt.show()

## "Average Completion Rate by Genre (%)"

_Completion rates are remarkably consistent across all genres, clustering tightly within a narrow 64.5% to 66.7% range—suggesting that genre alone is not a strong predictor of whether subscribers will finish content. Musical leads with the highest completion rate at 66.7%, followed closely by Stand-Up (66.3%) and Animation (66.2%), which may reflect the typically shorter runtime and highly engaging nature of these formats. The minimal variance across 16 genres—with only a 2.2 percentage point spread from top (Musical) to bottom (Reality at 64.5%)—indicates that viewer commitment is driven more by content quality, individual preferences, or platform-wide behavior patterns rather than genre-specific appeal._

# review sentiment breakown - a bar chart of Positive / Neutral / Negative review counts.

In [0]:
# 2. Count reviews by sentiment in a defined order
sentiment_order = ['Positive', 'Neutral', 'Negative']
sentiment_counts = (
    reviews_df['sentiment'].value_counts().reindex(sentiment_order)
)
percentages = (sentiment_counts / sentiment_counts.sum()) * 100

# 3. Create the Bar Chart
fig, ax = plt.subplots(figsize=(8, 5.5))

colors = ['#2a9d8f', '#e9c46a', '#e76f51']  # Green, Yellow, Coral
bars = ax.bar(
    sentiment_counts.index,
    sentiment_counts.values,
    color=colors,
    width=0.55,
    edgecolor='none',
)

# 4. Add count and percentage labels above each bar
for bar, pct in zip(bars, percentages):
  yval = bar.get_height()
  ax.text(
      bar.get_x() + bar.get_width() / 2,
      yval + 1200,
      f'{yval:,}\n({pct:.1f}%)',
      ha='center',
      va='bottom',
      fontsize=10,
      fontweight='bold',
      color='#333333',
  )

# 5. Styling and labels
ax.set_title(
    'Review Sentiment Breakdown', fontsize=14, fontweight='bold', pad=15
)
ax.set_xlabel('Sentiment', fontsize=11)
ax.set_ylabel('Number of Reviews', fontsize=11)
ax.set_ylim(0, sentiment_counts.max() * 1.15)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Insights for "Review Sentiment Breakdown" 

_Positive sentiment overwhelmingly dominates user feedback with 72,574 reviews (68.2%), nearly six times more than negative reviews (11,812, 11.1%), indicating strong overall subscriber satisfaction with the platform's content and service. Neutral reviews capture 20.7% (22,021), representing a moderate middle ground of users with mixed or reserved opinions. The combined positive and neutral sentiment accounts for 88.9% of all reviews, suggesting minimal dissatisfaction and validating that the platform is successfully meeting user expectations across content quality, user experience, and value proposition._